# Nik1 — train an on-device model family on a free Kaggle GPU

This notebook trains **nik1-route** (utterance → tool call) at a size that would
take too long on a laptop, and exports the same `.nik1` container the browser
runtime reads. Nothing about inference changes because the model got bigger.

**Setup:** `Runtime → Accelerator → GPU T4`, internet on. Then `Run all`.

| | laptop (this repo) | this notebook |
|---|---|---|
| params | 248k | 2–6M |
| int8 size | 254 KB | 2–6 MB |
| training | 4 min on 2 vCPU | ~15 min on a T4 |

In [ ]:
# 1. Get the code. Public repo, no token needed.
import os, subprocess, sys
REPO = "https://github.com/NikitHamal/Pixelforge.git"
BRANCH = "feat/nik1-on-device-models"
if not os.path.exists("/kaggle/working/Pixelforge"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/kaggle/working/Pixelforge"], check=True)
%cd /kaggle/working/Pixelforge
!git log --oneline -1
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2. Sanity: the data generator is pure Python and needs no torch.
!cd nik1/python && python3 -c "
import sys; sys.path.insert(0,'.')
from nik1.data import build_route_dataset
tr, va = build_route_dataset(seed=5, per_tool=40)
print('train', len(tr), 'val', len(va))
for e in tr[:3]: print(' ', e.text, '->', e.target)
"

In [ ]:
# 3. Train a bigger router on the GPU.
#    Roughly 2x the width and depth of the shipped model.
!cd nik1/python && python3 torch/train_router_torch.py \
    --d-model 256 --layers 6 --heads 8 --d-ff 768 \
    --steps 6000 --batch 64 --lr 1e-3 --vocab 2048 \
    --per-tool 600 --out /kaggle/working/nik1-out

In [ ]:
# 4. Look at what came out: size, metrics, a few free-running predictions.
import json, os
OUT = "/kaggle/working/nik1-out"
print(os.listdir(OUT))
m = json.load(open(f"{OUT}/nik1-route.metrics.json"))
print(json.dumps(m, indent=1))
for f in os.listdir(OUT):
    p = os.path.join(OUT, f)
    print(f"  {f:34} {os.path.getsize(p)/1024:8.1f} KB")

In [ ]:
# 5. Prove it in the runtime: drop the exported model in and run the JS tests.
import shutil, os
MODELS = "/kaggle/working/Pixelforge/nik1/js/models"
for f in ["nik1-route.nik1", "nik1-route.json"]:
    shutil.copy(f"/kaggle/working/nik1-out/{f}", os.path.join(MODELS, f))
!cd /kaggle/working/Pixelforge && node nik1/tools/sync-models.js && node nik1/tests/test_runtime.js --bench

In [ ]:
# 6. Download the artefacts (right-click the links, or use the Output tab).
from IPython.display import FileLink, display
for f in ["nik1-route.nik1", "nik1-route.json", "nik1-route.metrics.json"]:
    display(FileLink(f"/kaggle/working/nik1-out/{f}"))

### Using the trained model locally

```bash
cp /kaggle/working/nik1-out/nik1-route.nik1  nik1/js/models/
cp /kaggle/working/nik1-out/nik1-route.json  nik1/js/models/
node nik1/tools/sync-models.js
node nik1/tools/nik1-gate.js --fast
```

The runtime reads whatever size you trained — there is no architecture registry
to update, because the config travels inside the `.json` next to the weights.